# scGPT Embedding Extraction

This tutorial demonstrates how to extract cell embeddings using **scGPT** — a foundation model for single-cell data — which can be used as input to scE2TM.

## Model Checkpoint

We use the **continual pretrained** model checkpoint from scGPT, which is specifically designed for zero-shot cell embedding related tasks.

- **Model**: continual pretrained
- **Download**: [https://drive.google.com/drive/folders/1_GROJTzXiAV8HB4imruOTk6PEGuNOcgB](https://drive.google.com/drive/folders/1_GROJTzXiAV8HB4imruOTk6PEGuNOcgB)

## Output

The extracted embeddings are saved as `{dataset_name}.csv` in the `./data/` directory and will be automatically loaded by scE2TM when running the main demo.

## Installation

> **Note**: scGPT requires a specific environment with flash-attention and compatible CUDA versions. Please follow the official installation guide: [https://github.com/bowang-lab/scGPT](https://github.com/bowang-lab/scGPT)

In [ ]:
from pathlib import Path
import numpy as np
from scipy.stats import mode
import scanpy as sc
import warnings
import seaborn as sns
import pandas as pd
import sys
import os

# Set project root directory
PROJECT_ROOT = Path(__file__).parent.parent if '__file__' in dir() else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
EMBEDDING_DIR = PROJECT_ROOT / 'data'  # Store embeddings in same directory

# scGPT model path (adjust according to your setup)
SCGPT_MODEL_DIR = Path("/mnt/rao/home/chenhg/methods/scGPT/save/scGPT_human")

# Import scGPT
sys.path.insert(0, str(PROJECT_ROOT.parent))  # Adjust as needed
import scgpt as scg

# Check faiss availability
try:
    import faiss
    faiss_imported = True
except ImportError:
    faiss_imported = False
    print("faiss not installed! We highly recommend installing it for fast similarity search.")
    print("To install it, see https://github.com/facebookresearch/faiss/wiki/Installing-Faiss")

warnings.filterwarnings("ignore", category=ResourceWarning)

# Dataset name
name = 'Wang'

# Load data using internal path
data_csv_path = DATA_DIR / f'{name}_HIGHPRE.csv'
print(f"Loading data from: {data_csv_path}")

if not data_csv_path.exists():
    raise FileNotFoundError(f"Data file not found: {data_csv_path}")

data_csv = pd.read_csv(data_csv_path, sep=',', index_col=0)
data_adata = sc.AnnData(data_csv)

# Verify scGPT model directory exists
if not SCGPT_MODEL_DIR.exists():
    raise FileNotFoundError(f"scGPT model directory not found: {SCGPT_MODEL_DIR}")

# Extract embeddings using scGPT
gene_col = "gene_name"
data_adata.var["gene_name"] = data_adata.var_names

print(f"Embedding data with scGPT...")
ref_embed_adata = scg.tasks.embed_data(
    data_adata,
    SCGPT_MODEL_DIR,
    gene_col=gene_col,
    batch_size=64,
)

# Save embedding to CSV
output_path = EMBEDDING_DIR / f'{name}.csv'
ref_embed_adata.obsm["X_scGPT"].to_csv(output_path)
print(f"Embedding saved to: {output_path}")